In [ ]:
import numpy as np

#from google.colab import drive
#drive.mount('/content/drive')

diem      = np.load("/content/drive/MyDrive/python-learning/data/diem.npy")
thu_nhap  = np.load("/content/drive/MyDrive/python-learning/data/thu_nhap.npy")
du_no     = np.load("/content/drive/MyDrive/python-learning/data/du_no.npy")
nhom_no   = np.load("/content/drive/MyDrive/python-learning/data/nhom_no.npy")

Mounted at /content/drive


In [ ]:
import numpy as np

# 4 khách × 3 chỉ số
khach = np.array([[720, 25, 150],
                  [580, 15,  80],
                  [690, 30, 200],
                  [450, 40, 350]])   # shape (4, 3)

trong_so = np.array([0.5, 0.3, 0.2])  # shape (3,)

diem_tat_ca = khach @ trong_so   # shape (4,)
# 4 điểm rủi ro tính trong 1 lệnh
print(diem_tat_ca)

[397.5 310.5 394.  307. ]


In [ ]:
'''Load 3 array lên
Chuẩn hóa từng array về [0, 1] bằng Min-Max (ôn buổi 3)
Ghép thành ma trận shape (500, 3) bằng np.column_stack
Tính điểm rủi ro tổng hợp cho cả 500 khách bằng 1 dòng dot product với trọng số [0.5, 0.3, 0.2]
In: điểm TB, cao nhất, thấp nhất'''

import numpy as np

#1/load 3 arr
diem      = np.load("/content/drive/MyDrive/python-learning/data/diem.npy")
thu_nhap  = np.load("/content/drive/MyDrive/python-learning/data/thu_nhap.npy")
du_no     = np.load("/content/drive/MyDrive/python-learning/data/du_no.npy")

#2
min_max_diem = (diem - np.min(diem))/(np.max(diem) - np.min(diem))
min_max_thu_nhap = (thu_nhap - np.min(thu_nhap))/(np.max(thu_nhap) - np.min(thu_nhap))
min_max_du_no = (du_no - np.min(du_no))/(np.max(du_no) - np.min(du_no))

#print(min_max_diem)
#print(min_max_thu_nhap)
#print(min_max_du_no)

#3 chuẩn hoá từng arr
data = np.column_stack([diem, thu_nhap, du_no])
#tính min-max theo matrix shape (500,3)
min_data = np.min(data, axis = 0)
max_data = np.max(data, axis = 0)

min_max = (data - min_data)/(max_data - min_data)
#print(min_max)

#4 Tính điểm rủi ro tổng hợp cho cả 500 khách bằng 1 dòng dot product với trọng số [0.5, 0.3, 0.2]

trong_so = np.array([0.5, 0.3, 0.2])

risk_score = min_max @ trong_so
#print(risk_score)

#5 In: điểm TB, cao nhất, thấp nhất
print(f"Điểm TB: {np.mean(risk_score):.1f}")
print(f"Điểm cao nhất: {np.max(risk_score):.1f}")
print(f"Điểm thấp nhất: {np.min(risk_score):.1f}")

Điểm TB: 0.5
Điểm cao nhất: 0.8
Điểm thấp nhất: 0.2


In [ ]:
'''Bài 1 — Hệ thống tính điểm rủi ro đầy đủ
Viết hàm score_portfolio(data, weights):

Nhận ma trận data shape (n, m) và vector weights shape (m,)
Trả về vector điểm shape (n,)
Validate: weights phải tổng = 1, shape phải khớp
Xử lý lỗi bằng try/except (ôn lại từ M1)
Test với 500 khách từ buổi 1 (dùng min_max đã chuẩn hóa)'''
import numpy as np

def score_portfolio(data, weights):
    min_data = np.min(data, axis = 0)
    max_data = np.max(data, axis = 0)
    min_max = (data - min_data)/(max_data - min_data)
    try:
        if np.sum(weights) == 1:
            score = min_max @ weights
        return score
    except Exception as e:
        print(f"Lỗi không xác định: {e}")
        return None

data = np.column_stack([diem, thu_nhap, du_no])
trong_so = np.array([0.5, 0.3, 0.2])

test1 = score_portfolio(data, trong_so)
print(test1)

In [ ]:
'''Claude code chữa bài 1'''
'''code làm thiếu kiểm tra điều kiện: shape so sánh giữa data và weights; logic kiểm tra tổng trọng số = 1 chưa đúng'''
def score_portfolio(data, weights):
    try:
        # Validate shape
        if data.shape[1] != weights.shape[0]:
            raise ValueError(f"Shape không khớp: data có {data.shape[1]} cột, weights có {weights.shape[0]} phần tử")

        # Validate weights tổng = 1
        if not np.isclose(np.sum(weights), 1.0):
            raise ValueError(f"Weights phải tổng = 1, hiện tại = {np.sum(weights):.4f}")

        # Tính điểm
        min_data = np.min(data, axis=0)
        max_data = np.max(data, axis=0)
        min_max  = (data - min_data) / (max_data - min_data)
        return min_max @ weights

    except ValueError as e:
        print(f"Lỗi dữ liệu: {e}")
        return None
    except Exception as e:
        print(f"Lỗi không xác định: {e}")
        return None

data = np.column_stack([diem, thu_nhap, du_no])


test1 = score_portfolio(data, np.array([0.5, 0.3, 0.3]))
print(test1)



test2 = score_portfolio(data, np.array([0.5, 0.5]))
print(test2)

test3 = score_portfolio(data, np.array([0.5, 0.2, 0.3]))
print(test3)

In [ ]:
# Bước 1: Z-score từng cột
n = data.shape[0]
z = (data - np.mean(data, axis=0)) / np.std(data, axis=0)

# Bước 2: corr_matrix = Z.T @ Z / (n - 1)
corr_matrix = z.T @ z / (n - 1)

# Kiểm tra
print("Tự tính:")
print(np.round(corr_matrix, 4))

print("\nnp.corrcoef:")
print(np.round(np.corrcoef(data.T), 4))

Tự tính:
[[ 1.002  -0.0865  0.0244]
 [-0.0865  1.002   0.0298]
 [ 0.0244  0.0298  1.002 ]]

np.corrcoef:
[[ 1.     -0.0863  0.0244]
 [-0.0863  1.      0.0298]
 [ 0.0244  0.0298  1.    ]]


In [ ]:
# Bước 1: Z-score từng cột
n = data.shape[0]
z = (data - np.mean(data, axis=0)) / np.std(data, axis=0, ddof=1)

# Bước 2: corr_matrix = Z.T @ Z / (n - 1)
corr_matrix = z.T @ z / (n - 1)

# Kiểm tra
print("Tự tính:")
print(np.round(corr_matrix, 4))

print("\nnp.corrcoef:")
print(np.round(np.corrcoef(data.T), 4))

Tự tính:
[[ 1.     -0.0863  0.0244]
 [-0.0863  1.      0.0298]
 [ 0.0244  0.0298  1.    ]]

np.corrcoef:
[[ 1.     -0.0863  0.0244]
 [-0.0863  1.      0.0298]
 [ 0.0244  0.0298  1.    ]]


In [1]:
import numpy as np
cac_loai_vay = np.array([
    [8.5,  240, 0.7, 0.2],   # 0: vay mua nhà
    [12.0,  36, 0.0, 0.4],   # 1: vay tiêu dùng
    [9.0,  180, 0.6, 0.3],   # 2: vay mua xe
    [15.0,  12, 0.0, 0.7],   # 3: vay tín chấp
    [8.8,  240, 0.8, 0.2],   # 4: vay mua nhà thương mại
])
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

ss_01 = cosine_sim(cac_loai_vay[0],cac_loai_vay[1])
ss_02 = cosine_sim(cac_loai_vay[0],cac_loai_vay[2])
ss_03 = cosine_sim(cac_loai_vay[0],cac_loai_vay[3])
ss_04 = cosine_sim(cac_loai_vay[0],cac_loai_vay[4])



print(f"similarity giữa mua nhà vs tiêu dùng: {ss_01}")
print(f"similarity giữa mua nhà vs mua xe: {ss_02}")
print(f"similarity giữa mua nhà vs tín chấp: {ss_03}")
print(f"similarity giữa mua nhà vs mua nhà thương mại: {ss_04}")

ab = max(ss_01,ss_02,ss_03,ss_04)

print(f"similarity lớn nhất là: {ab}")


similarity giữa mua nhà vs tiêu dùng: 0.9592326687433683
similarity giữa mua nhà vs mua xe: 0.999893625110439
similarity giữa mua nhà vs tín chấp: 0.6515369709933415
similarity giữa mua nhà vs mua nhà thương mại: 0.9999991341515637
similarity lớn nhất là: 0.9999991341515637
